# VOICECLONE-QC RVC Bridge

Run the Drive authorization cell immediately after configuration. This notebook is pinned to the validated RVC v2 / 40k / RMVPE pipeline and saves resumable checkpoints to Drive.


In [ ]:
#@title 1. Configuration VOICECLONE-QC v1.3.0
from pathlib import Path

NOTEBOOK_VERSION = "1.3.0"
MODEL_NAME = "Alertes_Stephanie"  #@param {type:"string"}
RUN_MODE = "new"  #@param ["new", "resume"]
TARGET_SAMPLE_RATE = "40k"
MODEL_ARCHITECTURE = "v2"
PRETRAIN_TYPE = "OV2"
PITCH_METHOD = "rmvpe"
TOTAL_EPOCHS = 200  #@param {type:"integer"}
SAVE_FREQUENCY = 10  #@param {type:"integer"}
BATCH_SIZE = 7  #@param {type:"integer"}
CHECKPOINT_SYNC_SECONDS = 120  #@param {type:"integer"}

# Pin the exact RVC code and model assets used by this notebook.
RVC_REPOSITORY_COMMIT = "d618280cdef162c39bf74d03362592db5c41ad80"
TORCHCREPE_COMMIT = "19e2ec3d494c0797a5ff2a11408ec5838fba6681"
ORVC_REVISION = "425f8006582161af571e0d7f5ce646a535a14b65"

DRIVE_DATASET_DIR = "/content/drive/MyDrive/VOICECLONE_Datasets"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/RVC_Output"
DRIVE_TRAINING_DIR = "/content/drive/MyDrive/VOICECLONE_Training"
DRIVE_RUN_DIR = Path(DRIVE_TRAINING_DIR) / MODEL_NAME
DRIVE_EXPERIMENT_DIR = DRIVE_RUN_DIR / "experiment"
DRIVE_CHECKPOINT_DIR = DRIVE_RUN_DIR / "checkpoints"

ZIP_PATH = f"{DRIVE_DATASET_DIR}/{MODEL_NAME}_dataset_cleaned.zip"
NOW_DIR = "/content/Mangio-RVC-Fork"
EXP_DIR = f"{NOW_DIR}/logs/{MODEL_NAME}"
DATASET_DIR = f"/content/voiceclone_qc/{MODEL_NAME}/dataset"

if not MODEL_NAME.strip():
    raise ValueError("MODEL_NAME cannot be empty.")
import re
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", MODEL_NAME) or MODEL_NAME.endswith("."):
    raise ValueError("Use the same portable model name as the local application (letters, digits, _, -, .).")
TRAINING_COMPLETE = False
EXPORT_COMPLETE = False
if RUN_MODE not in {"new", "resume"}:
    raise ValueError("RUN_MODE must be 'new' or 'resume'.")
if (TARGET_SAMPLE_RATE, MODEL_ARCHITECTURE, PRETRAIN_TYPE) != ("40k", "v2", "OV2"):
    raise ValueError("This validated notebook is pinned to 40k, v2 and OV2 assets.")
if PITCH_METHOD != "rmvpe":
    raise ValueError("Use RMVPE to keep the production quality setting consistent.")
if TOTAL_EPOCHS < 1 or SAVE_FREQUENCY < 1 or BATCH_SIZE < 1:
    raise ValueError("Training values must be positive integers.")

print(
    f"VOICECLONE-QC RVC Bridge v{NOTEBOOK_VERSION} | "
    f"Model: {MODEL_NAME} | mode: {RUN_MODE} | target epochs: {TOTAL_EPOCHS}"
)


In [ ]:
#@title 2. Mount Google Drive - run this immediately
import os
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

for directory in (DRIVE_DATASET_DIR, DRIVE_OUTPUT_DIR, DRIVE_TRAINING_DIR, DRIVE_RUN_DIR):
    Path(directory).mkdir(parents=True, exist_ok=True)

print("Google Drive is ready. You can now leave Colab to continue the setup.")


In [ ]:
#@title VOICECLONE-QC - sauvegardes compactes et reprise verifiee
"""Embedded in the notebook by scripts/update_colab_notebook.py."""
from pathlib import Path
import hashlib
import json
import shutil
import tempfile
import uuid


def file_hash(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()


def write_json_atomic(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.write_text(json.dumps(data, indent=2), encoding="utf-8")
    temporary.replace(path)


def copy_verified(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected = file_hash(source)
    if destination.is_file() and file_hash(destination) == expected:
        return expected
    temporary = destination.with_name(destination.name + ".part")
    shutil.copy2(source, temporary)
    if file_hash(temporary) != expected:
        raise RuntimeError(f"Verification failed: {destination}")
    temporary.replace(destination)
    return expected


def checkpoint_epoch(path):
    import torch
    checkpoint = torch.load(str(path), map_location="cpu", weights_only=False)
    return int(checkpoint["iteration"])


class CheckpointStore:
    def __init__(self, run_dir, epoch_reader=checkpoint_epoch):
        self.root = Path(run_dir)
        self.experiment = self.root / "experiment"
        self.checkpoints = self.root / "checkpoints"
        self.pointer = self.checkpoints / "current.json"
        self.epoch_reader = epoch_reader
        self.last_signature = None

    def save_data(self, local):
        local = Path(local)
        self.experiment.mkdir(parents=True, exist_ok=True)
        files = {}
        for source in local.rglob("*"):
            if not source.is_file():
                continue
            relative = source.relative_to(local)
            if (source.name.startswith(("G_", "D_", "events.out.tfevents"))
                    or source.suffix in {".index", ".part"} or source.name == "total_fea.npy"):
                continue
            files[relative.as_posix()] = copy_verified(source, self.experiment / relative)
        write_json_atomic(self.root / "data_manifest.json", {"files": files})

    def save_checkpoints(self, local):
        local = Path(local)
        candidates = [sorted(local.glob(pattern), key=lambda p: p.stat().st_mtime_ns, reverse=True)
                      for pattern in ("G_*.pth", "D_*.pth")]
        if not all(candidates):
            return False
        pair = [items[0] for items in candidates]
        signature = tuple((p.name, p.stat().st_size, p.stat().st_mtime_ns) for p in pair)
        if signature == self.last_signature:
            return True
        # Freeze both local files before uploading. RVC may be writing the next save.
        with tempfile.TemporaryDirectory(prefix="voiceclone-checkpoint-") as scratch:
            frozen = [Path(scratch) / p.name for p in pair]
            try:
                for source, target in zip(pair, frozen):
                    shutil.copy2(source, target)
                if signature != tuple((p.name, p.stat().st_size, p.stat().st_mtime_ns) for p in pair):
                    return False
                epochs = [self.epoch_reader(p) for p in frozen]
                if epochs[0] != epochs[1]:
                    return False
            except Exception as error:
                print(f"Checkpoint still being written or unreadable; previous backup retained: {type(error).__name__}")
                return False
            generation = "generation_" + uuid.uuid4().hex
            target = self.checkpoints / generation
            hashes = {p.name: copy_verified(p, target / p.name) for p in frozen}
            manifest_path = self.root / "data_manifest.json"
            manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {"files": {}}
            for name in ("config.json", "filelist.txt"):
                if (local / name).is_file():
                    manifest["files"][name] = copy_verified(local / name, self.experiment / name)
            write_json_atomic(manifest_path, manifest)
            # The old complete pair remains valid until the new pair is verified.
            write_json_atomic(self.pointer, {"generation": generation, "epoch": epochs[0], "files": hashes})
            self.last_signature = signature
        # Retain one committed pair, never two permanent copies in experiment/.
        for folder in self.checkpoints.glob("generation_*"):
            if folder.name != generation and folder.is_dir() and not folder.is_symlink():
                shutil.rmtree(folder)
        for folder in (self.checkpoints, self.experiment):
            for pattern in ("G_*.pth", "D_*.pth"):
                for obsolete in folder.glob(pattern):
                    obsolete.unlink()
        print(f"Verified checkpoint backup: epoch {epochs[0]}", flush=True)
        return True

    def restore(self, local):
        local = Path(local)
        if not self.experiment.is_dir():
            raise RuntimeError("Training data backup is missing. Restore the local archive to Drive first.")
        if local.exists():
            shutil.rmtree(local)
        shutil.copytree(self.experiment, local)
        manifest = self.root / "data_manifest.json"
        if manifest.exists():
            for name, expected in json.loads(manifest.read_text())["files"].items():
                relative = Path(name)
                if relative.is_absolute() or ".." in relative.parts:
                    raise RuntimeError("Invalid backup data path")
                if file_hash(local / relative) != expected:
                    raise RuntimeError(f"Corrupt training data: {name}")
        if self.pointer.exists():
            data = json.loads(self.pointer.read_text())
            generation = data["generation"]
            if not generation.startswith("generation_") or Path(generation).name != generation:
                raise RuntimeError("Invalid checkpoint generation")
            pair = []
            for name, expected in data["files"].items():
                if Path(name).name != name or not name.startswith(("G_", "D_")) or not name.endswith(".pth"):
                    raise RuntimeError("Invalid checkpoint filename")
                source = self.checkpoints / generation / name
                if file_hash(source) != expected:
                    raise RuntimeError(f"Corrupt checkpoint: {name}")
                pair.append(source)
        else:
            # v1.2.x stored newer checkpoints beside an older experiment copy.
            by_epoch = {}
            for folder in (self.experiment, self.checkpoints):
                for pattern in ("G_*.pth", "D_*.pth"):
                    for candidate in folder.glob(pattern):
                        try:
                            epoch = self.epoch_reader(candidate)
                            by_epoch.setdefault(epoch, {})[candidate.name[0]] = candidate
                        except Exception:
                            continue
            valid = [epoch for epoch, items in by_epoch.items() if set(items) == {"G", "D"}]
            if not valid:
                raise RuntimeError("No complete G/D checkpoint pair. Resume refused instead of restarting at zero.")
            pair = list(by_epoch[max(valid)].values())
        if len(pair) != 2 or {p.name[0] for p in pair} != {"G", "D"}:
            raise RuntimeError("Incomplete checkpoint pair")
        epochs = [self.epoch_reader(p) for p in pair]
        if epochs[0] != epochs[1]:
            raise RuntimeError("G and D checkpoints have different epochs")
        for pattern in ("G_*.pth", "D_*.pth"):
            for stale in local.glob(pattern):
                stale.unlink()
        for source in pair:
            copy_verified(source, local / source.name)
        print(f"Restored verified checkpoint pair: epoch {epochs[0]}")
        return epochs[0]


In [ ]:
#@title 3. Install Dependencies (Colab Python 3.12 compatible)
import os
import subprocess
import sys

# A failed/restarted repository cell can leave the process in a deleted folder.
# pip calls os.getcwd(), so always restore a valid Colab working directory first.
os.chdir("/content")

system_packages = ["build-essential", "python3-dev", "ffmpeg", "aria2"]
python_packages = [
    "faiss-cpu",
    "ffmpeg-python",
    "praat-parselmouth",
    "pyworld",
    "numpy",
    "numba",
    "librosa",
    "tensorboardX",
    "tensorboard",
    "onnx",
    "onnxruntime-gpu",
    "torchcrepe",
    "python-dotenv",
    "av",
    "scikit-learn",
]

def run_command(command, label):
    print(f"Installing: {label}", flush=True)
    completed = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.returncode != 0:
        raise RuntimeError(
            f"INSTALLATION FAILED: {label} (exit code {completed.returncode})."
        )

print("Updating package list...", flush=True)
run_command(["apt-get", "update", "-qq"], "system package list")

for package in system_packages:
    run_command(["apt-get", "install", "-qq", "-y", package], f"system package {package}")

# Colab owns its system pip. Do not upgrade pip, setuptools, or wheel here.
# The flag is required by the current Python 3.12 Debian environment.
pip_prefix = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--disable-pip-version-check",
    "--no-input",
    "--prefer-binary",
    "--break-system-packages",
    "--upgrade",
]

for package in python_packages:
    run_command(pip_prefix + [package], f"Python package {package}")

run_command(pip_prefix + ["fairseq-fixed"], "Python package fairseq-fixed")
print("Dependencies ready.", flush=True)


In [ ]:
#@title 4. Download RVC Source Code (pinned)
import os
import shutil
import subprocess
import zipfile

# Never delete a repository while Python is still using it as the current path.
os.chdir("/content")

def download_github_archive(repository, revision, destination):
    archive_path = Path("/content") / f"{destination.name}-{revision[:12]}.zip"
    extract_dir = Path("/content") / f"{destination.name}-{revision[:12]}-extract"
    for path in (destination, archive_path, extract_dir):
        if path.exists():
            shutil.rmtree(path) if path.is_dir() else path.unlink()

    url = f"https://github.com/{repository}/archive/{revision}.zip"
    completed = subprocess.run(
        [
            "curl", "--fail", "--location", "--retry", "5", "--retry-all-errors",
            "--connect-timeout", "30", "--output", str(archive_path), url,
        ],
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f"Unable to download {repository} at {revision[:12]}.\n"
            f"curl output:\n{completed.stderr or completed.stdout}"
        )
    with zipfile.ZipFile(archive_path, "r") as archive:
        archive.extractall(extract_dir)
    source_dirs = [path for path in extract_dir.iterdir() if path.is_dir()]
    if len(source_dirs) != 1:
        raise RuntimeError(f"Unexpected archive layout for {repository}: {source_dirs}")
    shutil.move(str(source_dirs[0]), str(destination))
    archive_path.unlink()
    shutil.rmtree(extract_dir)

repo_dir = Path("/content/Mangio-RVC-Fork")
torchcrepe_dir = Path("/content/torchcrepe")
download_github_archive("Mangio621/Mangio-RVC-Fork", RVC_REPOSITORY_COMMIT, repo_dir)
download_github_archive("maxrmorrison/torchcrepe", TORCHCREPE_COMMIT, torchcrepe_dir)
shutil.copytree(torchcrepe_dir / "torchcrepe", repo_dir / "torchcrepe", dirs_exist_ok=True)
os.chdir(repo_dir)
print(f"RVC source ready: {repo_dir} @ {RVC_REPOSITORY_COMMIT[:12]}")


In [ ]:
#@title 5. GPU Check
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Change runtime type to GPU.')

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_properties(i).total_memory // 1024 // 1024, 'MB')

gpus = '-'.join(str(i) for i in range(torch.cuda.device_count()))
print('Using GPU ids:', gpus)


In [ ]:
#@title 6. Download verified pretrained models and RMVPE
import hashlib
import json
import subprocess

assets = {
    Path(NOW_DIR) / "pretrained_v2" / "f0G40k_OV2.pth": [
        f"https://huggingface.co/ORVC/Ov2Super/resolve/{ORVC_REVISION}/f0Ov2Super40kG.pth",
        "https://huggingface.co/ORVC/Ov2Super/resolve/main/f0Ov2Super40kG.pth",
    ],
    Path(NOW_DIR) / "pretrained_v2" / "f0D40k_OV2.pth": [
        f"https://huggingface.co/ORVC/Ov2Super/resolve/{ORVC_REVISION}/f0Ov2Super40kD.pth",
        "https://huggingface.co/ORVC/Ov2Super/resolve/main/f0Ov2Super40kD.pth",
    ],
    Path(NOW_DIR) / "hubert_base.pt": [
        "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt",
    ],
    Path(NOW_DIR) / "rmvpe.pt": [
        "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt",
    ],
    Path(NOW_DIR) / "configs" / "40k.json": [
        f"https://raw.githubusercontent.com/Mangio621/Mangio-RVC-Fork/{RVC_REPOSITORY_COMMIT}/configs/40k.json",
    ],
}

def download_verified(destination, urls):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    failures = []
    for url in urls:
        if temporary.exists():
            temporary.unlink()
        try:
            subprocess.check_call([
                "curl", "--fail", "--location", "--retry", "5", "--retry-all-errors",
                "--connect-timeout", "30", "--output", str(temporary), url,
            ])
            if temporary.stat().st_size < 100:
                raise RuntimeError("downloaded file is unexpectedly small")
            temporary.replace(destination)
            return url, hashlib.sha256(destination.read_bytes()).hexdigest()
        except Exception as error:
            failures.append(f"{url}: {error}")
    raise RuntimeError(
        f"Unable to download {destination.name}. Tried:\n" + "\n".join(failures)
    )

manifest = {}
for destination, urls in assets.items():
    used_url, checksum = download_verified(destination, urls)
    manifest[str(destination.relative_to(NOW_DIR))] = {
        "url": used_url,
        "sha256": checksum,
        "bytes": destination.stat().st_size,
    }

manifest_path = Path(NOW_DIR) / "voiceclone_asset_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Verified {len(manifest)} assets: {manifest_path}")


In [ ]:
#@title 7. Restore a resumable training run from Drive
import shutil

local_experiment = Path(EXP_DIR)
checkpoint_store = CheckpointStore(DRIVE_RUN_DIR)
if RUN_MODE == "resume":
    restored_epoch = checkpoint_store.restore(local_experiment)
    if TOTAL_EPOCHS <= restored_epoch:
        raise ValueError(f"TOTAL_EPOCHS must be greater than the restored epoch {restored_epoch}.")
else:
    if any(path.is_file() for path in DRIVE_RUN_DIR.rglob("*")):
        raise RuntimeError("An existing training backup uses this name. Archive and release it locally, use resume, or choose a new name. Nothing was overwritten.")
    if local_experiment.exists():
        shutil.rmtree(local_experiment)
    print("New run selected. Compact verified backups enabled.")


In [ ]:
#@title 8. Load VOICECLONE-QC Dataset ZIP
import os
import shutil
import zipfile

if RUN_MODE == "resume":
    print("Resume mode: dataset import is skipped.")
else:
    dataset_dir = Path(DATASET_DIR)
    if not Path(ZIP_PATH).exists():
        raise FileNotFoundError(f"Dataset ZIP not found: {ZIP_PATH}")
    if dataset_dir.parent.exists():
        shutil.rmtree(dataset_dir.parent)
    dataset_dir.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        archive.extractall(dataset_dir.parent)
    readme = dataset_dir / "README.txt"
    if readme.exists():
        readme.unlink()
    wav_files = sorted(dataset_dir.glob("*.wav"))
    if not wav_files:
        raise RuntimeError(f"No WAV files found in {dataset_dir}")
    print(f"Dataset ready: {dataset_dir} ({len(wav_files)} WAV files)")


In [ ]:
#@title 9. Setup CSVDB
import os
import csv

os.chdir(NOW_DIR)
os.makedirs('csvdb', exist_ok=True)

with open('csvdb/formanting.csv', 'w', newline='') as frmnt:
    csv.writer(frmnt, delimiter=',').writerow([False, 1.0, 1.0])

with open('csvdb/stop.csv', 'w', newline='') as stp:
    csv.writer(stp, delimiter=',').writerow([False])

DoFormant, Quefrency, Timbre = False, 1.0, 1.0
print('CSVDB ready:', os.path.abspath('csvdb/formanting.csv'))


In [ ]:
#@title 10. Preprocess Dataset
import os
import subprocess

if RUN_MODE == "resume":
    print("Resume mode: preprocessing is skipped.")
else:
    cpu_threads = min(2, os.cpu_count() or 1)
    Path(EXP_DIR).mkdir(parents=True, exist_ok=True)
    command = [
        "python", "trainset_preprocess_pipeline_print.py", DATASET_DIR, "40000",
        str(cpu_threads), EXP_DIR, "1",
    ]
    print(" ".join(command))
    subprocess.check_call(command)
    print("Preprocessing complete.")


In [ ]:
#@title 11. Python 3.12 / PyTorch Compatibility Patch
from pathlib import Path

sitecustomize = Path(NOW_DIR) / "sitecustomize.py"
sitecustomize.write_text(r'''
import pkgutil
import importlib.machinery

if not hasattr(importlib.machinery.FileFinder, "find_module"):
    def _voiceclone_find_module(self, fullname, path=None):
        spec = self.find_spec(fullname)
        return None if spec is None else spec.loader
    importlib.machinery.FileFinder.find_module = _voiceclone_find_module

if not hasattr(pkgutil, "ImpImporter"):
    class ImpImporter:
        def __init__(self, *args, **kwargs):
            pass
        def find_module(self, fullname, path=None):
            return None
    pkgutil.ImpImporter = ImpImporter

if not hasattr(pkgutil, "ImpLoader"):
    class ImpLoader:
        pass
    pkgutil.ImpLoader = ImpLoader

try:
    import torch
    _voiceclone_original_torch_load = torch.load

    def _voiceclone_torch_load_compat(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _voiceclone_original_torch_load(*args, **kwargs)

    torch.load = _voiceclone_torch_load_compat
except Exception:
    pass
''', encoding="utf-8")

print("Compatibility patch written:", sitecustomize)


In [ ]:
#@title 12. Feature Extraction RMVPE
import os
import subprocess

if RUN_MODE == "resume":
    print("Resume mode: RMVPE and HuBERT extraction are skipped.")
else:
    cpu_threads = min(2, os.cpu_count() or 1)
    environment = os.environ.copy()
    environment["PYTHONPATH"] = f"{NOW_DIR}:{environment.get('PYTHONPATH', '')}"
    f0_command = ["python", "extract_f0_print.py", EXP_DIR, str(cpu_threads), "rmvpe", "128"]
    feature_command = ["python", "extract_feature_print.py", "device", "1", "0", "0", EXP_DIR, "v2"]
    print(" ".join(f0_command))
    subprocess.check_call(f0_command, env=environment)
    print(" ".join(feature_command))
    subprocess.check_call(feature_command, env=environment)
    print("Feature extraction complete.")


In [ ]:
#@title 12b. Matplotlib / NumPy Compatibility Patch
from pathlib import Path

utils_path = Path(NOW_DIR) / "train" / "utils.py"
text = utils_path.read_text(encoding="utf-8")

if "def _voiceclone_canvas_tostring_rgb" not in text:
    marker = "import matplotlib.pyplot as plt\n"
    patch = '''import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg

if not hasattr(FigureCanvasAgg, "tostring_rgb"):
    def _voiceclone_canvas_tostring_rgb(self):
        return self.buffer_rgba().tobytes()
    FigureCanvasAgg.tostring_rgb = _voiceclone_canvas_tostring_rgb
'''
    if marker in text:
        text = text.replace(marker, patch, 1)
    else:
        text = patch + "\n" + text

text = text.replace(
    'np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep="")',
    'np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)'
)

text = text.replace(
    "np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep='')",
    "np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)"
)

utils_path.write_text(text, encoding="utf-8")
print("Patched:", utils_path)


In [ ]:
#@title 12c. RGB Canvas Patch
from pathlib import Path

utils_path = Path(NOW_DIR) / "train" / "utils.py"
text = utils_path.read_text(encoding="utf-8")

text = text.replace(
    "return self.buffer_rgba().tobytes()",
    "import numpy as _np\n        return _np.asarray(self.buffer_rgba())[:, :, :3].tobytes()"
)

utils_path.write_text(text, encoding="utf-8")
print("RGB patch OK:", utils_path)


In [ ]:
#@title 13. Save training data once and keep one complete checkpoint pair
def sync_checkpoints_to_drive():
    return checkpoint_store.save_checkpoints(Path(EXP_DIR))

def sync_experiment_to_drive():
    checkpoint_store.save_data(Path(EXP_DIR))
    sync_checkpoints_to_drive()
    print(f"Training data verified without duplicate model weights: {DRIVE_EXPERIMENT_DIR}")

if RUN_MODE == "new":
    sync_experiment_to_drive()
else:
    print("Resume mode: the restored backup remains the source of truth.")


In [ ]:
#@title 14. Train RVC Model with automatic checkpoint backups
import math
import os
import subprocess
import time
from random import shuffle

os.chdir(NOW_DIR)
# Build the exact RVC file list expected by train_nsf_sim_cache_sid_load_pretrain.py.
# This must happen after preprocessing and feature extraction.
gt_wavs_dir = f"{EXP_DIR}/0_gt_wavs"
feature_dir = f"{EXP_DIR}/3_feature768"
f0_dir = f"{EXP_DIR}/2a_f0"
f0nsf_dir = f"{EXP_DIR}/2b-f0nsf"
required_dirs = [gt_wavs_dir, feature_dir, f0_dir, f0nsf_dir]
missing_dirs = [path for path in required_dirs if not Path(path).is_dir()]
if missing_dirs:
    raise FileNotFoundError(
        "Training data is incomplete. Run preprocessing and RMVPE feature extraction first.\n"
        + "\n".join(missing_dirs)
    )

speaker_id = 0
names = (
    {name.split(".")[0] for name in os.listdir(gt_wavs_dir)}
    & {name.split(".")[0] for name in os.listdir(feature_dir)}
    & {name.split(".")[0] for name in os.listdir(f0_dir)}
    & {name.split(".")[0] for name in os.listdir(f0nsf_dir)}
)
if not names:
    raise RuntimeError("No matching WAV, feature, F0, and F0NSF files were found for training.")

filelist = [
    f"{gt_wavs_dir}/{name}.wav|{feature_dir}/{name}.npy|"
    f"{f0_dir}/{name}.wav.npy|{f0nsf_dir}/{name}.wav.npy|{speaker_id}"
    for name in names
]
for _ in range(2):
    filelist.append(
        f"{NOW_DIR}/logs/mute/0_gt_wavs/mute40k.wav|"
        f"{NOW_DIR}/logs/mute/3_feature768/mute.npy|"
        f"{NOW_DIR}/logs/mute/2a_f0/mute.wav.npy|"
        f"{NOW_DIR}/logs/mute/2b-f0nsf/mute.wav.npy|{speaker_id}"
    )
shuffle(filelist)
filelist_path = Path(EXP_DIR) / "filelist.txt"
filelist_path.write_text("\n".join(filelist), encoding="utf-8")
if filelist_path.stat().st_size == 0:
    raise RuntimeError("filelist.txt was created empty; training was not started.")
print(f"Filelist written: {len(filelist)} items -> {filelist_path}")
TRAINING_COMPLETE = False
EXPORT_COMPLETE = False
checkpoint_store.save_data(Path(EXP_DIR))

sync_checkpoints_to_drive()
environment = os.environ.copy()
environment["PYTHONPATH"] = f"{NOW_DIR}:{environment.get('PYTHONPATH', '')}"
command = [
    "python", "train_nsf_sim_cache_sid_load_pretrain.py",
    "-e", MODEL_NAME, "-sr", "40k", "-f0", "1", "-bs", str(BATCH_SIZE),
    "-g", "0", "-te", str(TOTAL_EPOCHS), "-se", str(SAVE_FREQUENCY),
    "-pg", "pretrained_v2/f0G40k_OV2.pth", "-pd", "pretrained_v2/f0D40k_OV2.pth",
    "-l", "1", "-c", "0", "-sw", "1", "-v", "v2", "-li", "3",
]
print(" ".join(command))
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=environment,
)
last_sync = time.monotonic()
saw_traceback = False
for line in process.stdout:
    print(line, end="")
    if "Traceback (most recent call last):" in line:
        saw_traceback = True
    if time.monotonic() - last_sync >= CHECKPOINT_SYNC_SECONDS:
        sync_checkpoints_to_drive()
        last_sync = time.monotonic()
exit_code = process.wait()
sync_checkpoints_to_drive()
if exit_code != 0 or saw_traceback:
    raise RuntimeError(
        f"Training failed (exit code {exit_code}). Checkpoints remain on Drive. "
        "Read the traceback printed above; do not proceed to index or export."
    )
sync_experiment_to_drive()
if not sync_checkpoints_to_drive():
    raise RuntimeError("Final checkpoint backup is incomplete; export refused.")
TRAINING_COMPLETE = True
print("Training complete and fully backed up to Drive.")


In [ ]:
#@title 15. Train Index
import os
import sys
import traceback
import numpy as np
import faiss

now_dir = NOW_DIR
experiment_name = MODEL_NAME
model_architecture = MODEL_ARCHITECTURE
exp_dir = EXP_DIR
feature_dir = f'{exp_dir}/3_feature256' if model_architecture == 'v1' else f'{exp_dir}/3_feature768'

if not os.path.exists(feature_dir):
    raise Exception('No features exist. Run Feature Extraction first.')
files = sorted(os.listdir(feature_dir))
if not files:
    raise Exception('No feature files found. Run Feature Extraction first.')

try:
    from sklearn.cluster import MiniBatchKMeans
except Exception:
    MiniBatchKMeans = None

npys = [np.load(f'{feature_dir}/{name}') for name in files]
big_npy = np.concatenate(npys, 0)
np.random.shuffle(big_npy)

if big_npy.shape[0] > 2e5 and MiniBatchKMeans is not None:
    print('KMeans reduction:', big_npy.shape)
    big_npy = MiniBatchKMeans(
        n_clusters=10000, verbose=True, batch_size=256,
        compute_labels=False, init='random'
    ).fit(big_npy).cluster_centers_

np.save(f'{exp_dir}/total_fea.npy', big_npy)
n_ivf = max(1, min(int(16 * np.sqrt(big_npy.shape[0])), max(1, big_npy.shape[0] // 39)))
print('Index shape:', big_npy.shape, 'n_ivf:', n_ivf)

index = faiss.index_factory(256 if model_architecture == 'v1' else 768, f'IVF{n_ivf},Flat')
index_ivf = faiss.extract_index_ivf(index)
index_ivf.nprobe = 1
print('Training index...')
index.train(big_npy)
faiss.write_index(index, f'{exp_dir}/trained_IVF{n_ivf}_Flat_nprobe_{index_ivf.nprobe}_{experiment_name}_{model_architecture}.index')
print('Adding vectors...')
for i in range(0, big_npy.shape[0], 8192):
    index.add(big_npy[i:i+8192])
index_path = f'{exp_dir}/added_IVF{n_ivf}_Flat_nprobe_{index_ivf.nprobe}_{experiment_name}_{model_architecture}.index'
faiss.write_index(index, index_path)
print('Index ready:', index_path)


In [ ]:
#@title 16. Export model and index to RVC_Output
import glob
import os
import shutil
from datetime import datetime

EXPORT_COMPLETE = False
if not TRAINING_COMPLETE:
    raise RuntimeError("Complete the training cell successfully before exporting.")

# RVC writes its final, inference-ready model to weights/. The G_*.pth and
# D_*.pth files in logs/ are training checkpoints and must not be copied as the
# final model. Some Mangio RVC runs need the official extractor called here.
weights_dir = Path(NOW_DIR) / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)
expected_weight = weights_dir / f"{MODEL_NAME}.pth"
weight_candidates = [Path(NOW_DIR) / "weights" / f"{MODEL_NAME}.pth"]
weight_candidates += [
    Path(path) for path in glob.glob(f"{NOW_DIR}/weights/{MODEL_NAME}_*.pth")
]
weight_candidates = [path for path in weight_candidates if path.exists()]
if not weight_candidates:
    checkpoint_candidates = [Path(path) for path in glob.glob(f"{EXP_DIR}/G_*.pth")]
    if not checkpoint_candidates:
        raise FileNotFoundError(
            "No G_*.pth training checkpoint found. Complete the training cell first."
        )
    latest_checkpoint = max(checkpoint_candidates, key=lambda item: item.stat().st_mtime)
    os.chdir(NOW_DIR)
    from train.process_ckpt import extract_small_model

    extraction_result = extract_small_model(
        str(latest_checkpoint),
        MODEL_NAME,
        "40k",
        1,
        f"VOICECLONE-QC export from {latest_checkpoint.name}",
        "v2",
    )
    if extraction_result != "Success." or not expected_weight.exists():
        raise RuntimeError(
            "Unable to create the final inference model from the latest G checkpoint: "
            f"{extraction_result}"
        )
    weight_candidates = [expected_weight]
    print(f"Final inference model created from: {latest_checkpoint}")

index_candidates = [Path(path) for path in glob.glob(f"{EXP_DIR}/added_*.index")]
if not index_candidates:
    raise FileNotFoundError("No added_*.index file found. Run the index cell first.")

source_model = max(weight_candidates, key=lambda item: item.stat().st_mtime)
source_index = max(index_candidates, key=lambda item: item.stat().st_mtime)
destination_model = Path(DRIVE_OUTPUT_DIR) / f"{MODEL_NAME}.pth"
destination_index = Path(DRIVE_OUTPUT_DIR) / f"{MODEL_NAME}.index"
if destination_model.exists() or destination_index.exists():
    archive = Path(DRIVE_OUTPUT_DIR) / "archive" / MODEL_NAME / datetime.now().strftime("%Y%m%d_%H%M%S")
    archive.mkdir(parents=True, exist_ok=True)
    for existing in (destination_model, destination_index):
        if existing.exists():
            shutil.copy2(existing, archive / existing.name)
    print(f"Previous export archived: {archive}")

copy_verified(source_model, destination_model)
copy_verified(source_index, destination_index)
if destination_model.stat().st_size == 0 or destination_index.stat().st_size == 0:
    raise RuntimeError("Export verification failed: an output file is empty.")
print(f"Exported model: {destination_model}")
print(f"Exported index: {destination_index}")
EXPORT_COMPLETE = True
print("Next: retrieve the model locally, stop Colab, archive Drive locally, then release Drive from the application.")


In [ ]:
#@title 17. Auto-disconnect runtime after successful export
from pathlib import Path
from google.colab import runtime
import time

if not globals().get("EXPORT_COMPLETE", False):
    raise RuntimeError("Current export not verified; runtime remains connected.")

required_files = [
    Path(DRIVE_OUTPUT_DIR) / f"{MODEL_NAME}.pth",
    Path(DRIVE_OUTPUT_DIR) / f"{MODEL_NAME}.index",
]

missing = [str(path) for path in required_files if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Export incomplet. Le runtime reste connecte.\nManquant:\n" + "\n".join(missing)
    )

print("Export confirme:")
for path in required_files:
    print(path)

print("Deconnexion du runtime dans 10 secondes...")
time.sleep(10)

runtime.unassign()
